# Notebook 07 — Preparation Pipeline

This notebook performs the full preprocessing pipeline for the Construction Delay Risk Prediction System (CDRPS).  
It applies all preparation steps to the raw dataset before modeling.

The preparation pipeline includes:

- Cleaning the dataset (handling missing values, fixing types, removing invalid rows)
- Encoding categorical variables
- Scaling numerical variables
- Splitting the dataset into train, validation, and test sets

This notebook ensures that the dataset is transformed into a clean, consistent, and model‑ready format.  
All transformations are performed using the modular functions defined in the `src/preparation` package.


In [45]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:", project_root)


Project root added: c:\Users\olive\Desktop\My ML Material\CONSTRUCTION DELAY RISK PREDICTION SYSTEM\CDRPS


In [46]:
import pandas as pd

from src.ingestion.file_loader import load_file
from src.preparation.cleaning import clean_dataset
from src.preparation.encoding import encode_categorical, scale_numerical
from src.preparation.splitter import split_dataset


In [47]:
# === Configuration from Notebook 04 ===

numeric_columns = [
    # example:
    # "Planned_Duration",
    # "Actual_Duration",
    # "Budget",
]

categorical_columns = [
    # "Contractor_Type",
    # "Region",
]

datetime_columns = [
    # "Start_Date",
    # "End_Date",
]

non_negative_columns = [
    # "Planned_Duration",
    # "Actual_Duration",
]

target_column = "Respondant Information.3"

test_size = 0.2
val_size = 0.1
random_state = 42
use_stratify = True

print("Configuration loaded.")


Configuration loaded.


In [48]:
df_raw = load_file(r"C:\Users\olive\Desktop\My ML Material\CONSTRUCTION DELAY RISK PREDICTION SYSTEM\archive(1)\Road Constuction Delay Survey.csv")

# Normalize column names and remove the survey-header row captured as data.
df_raw.columns = [col.strip() if isinstance(col, str) else col for col in df_raw.columns]
if "Respondent Number" in df_raw.columns:
    respondent_num = pd.to_numeric(df_raw["Respondent Number"], errors="coerce")
    df_raw = df_raw[respondent_num.notna()].copy()

print("Dataset loaded successfully.")
df_raw.head()


Dataset loaded successfully.


,Respondent Number,Category 1: Materials,Category 1: Materials.1,Category 1: Materials.2,Category 1: Materials.3,Category 1: Materials.4,Category 2: Labor and Equipment,Category 2: Labor and Equipment.1,Category 2: Labor and Equipment.2,Category 2: Labor and Equipment.3,...,Category 8: Scope of work.3,Category 9: External issues,Category 9: External issues .1,Category 9: External issues .2,Respondant Information,Respondant Information.1,Respondant Information.2,Respondant Information.3,Respondant Information.4,Respondant Information.5
1,1.0,1,1,1,1,1,1,1,1,1,...,1,2,2,3,1,2,2,2,3,5
2,2.0,2,5,3,2,3,2,2,2,3,...,3,1,1,4,1,3,2,3,3,3
3,3.0,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,2,1,2,4
4,4.0,2,2,3,3,2,2,3,4,4,...,3,4,3,3,1,3,1,1,3,4
5,5.0,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,3,1,2,4


In [49]:
df_clean = clean_dataset(
    df_raw,
    numeric_columns=numeric_columns,
    datetime_columns=datetime_columns,
    non_negative_columns=non_negative_columns
)

print("Cleaning completed.")
df_clean.head()


Cleaning completed.


,Respondent Number,Category 1: Materials,Category 1: Materials.1,Category 1: Materials.2,Category 1: Materials.3,Category 1: Materials.4,Category 2: Labor and Equipment,Category 2: Labor and Equipment.1,Category 2: Labor and Equipment.2,Category 2: Labor and Equipment.3,...,Category 8: Scope of work.3,Category 9: External issues,Category 9: External issues .1,Category 9: External issues .2,Respondant Information,Respondant Information.1,Respondant Information.2,Respondant Information.3,Respondant Information.4,Respondant Information.5
1,1.0,1,1,1,1,1,1,1,1,1,...,1,2,2,3,1,2,2,2,3,5
2,2.0,2,5,3,2,3,2,2,2,3,...,3,1,1,4,1,3,2,3,3,3
3,3.0,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,2,1,2,4
4,4.0,2,2,3,3,2,2,3,4,4,...,3,4,3,3,1,3,1,1,3,4
5,5.0,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,3,1,2,4


In [50]:
df_encoded, encoding_metadata = encode_categorical(
    df_clean,
    categorical_columns=categorical_columns
)

print("Categorical encoding completed.")
df_encoded.head()


Categorical encoding completed.


,Respondent Number,Category 1: Materials,Category 1: Materials.1,Category 1: Materials.2,Category 1: Materials.3,Category 1: Materials.4,Category 2: Labor and Equipment,Category 2: Labor and Equipment.1,Category 2: Labor and Equipment.2,Category 2: Labor and Equipment.3,...,Category 8: Scope of work.3,Category 9: External issues,Category 9: External issues .1,Category 9: External issues .2,Respondant Information,Respondant Information.1,Respondant Information.2,Respondant Information.3,Respondant Information.4,Respondant Information.5
1,1.0,1,1,1,1,1,1,1,1,1,...,1,2,2,3,1,2,2,2,3,5
2,2.0,2,5,3,2,3,2,2,2,3,...,3,1,1,4,1,3,2,3,3,3
3,3.0,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,2,1,2,4
4,4.0,2,2,3,3,2,2,3,4,4,...,3,4,3,3,1,3,1,1,3,4
5,5.0,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,3,1,2,4


In [51]:
print("numeric_columns:", numeric_columns)
print("Columns in df_encoded:", df_encoded.columns.tolist())

missing = [col for col in numeric_columns if col not in df_encoded.columns]
print("Missing numeric columns:", missing)


numeric_columns: []
Columns in df_encoded: ['Respondent Number', 'Category 1: Materials', 'Category 1: Materials.1', 'Category 1: Materials.2', 'Category 1: Materials.3', 'Category 1: Materials.4', 'Category 2: Labor and Equipment', 'Category 2: Labor and Equipment.1', 'Category 2: Labor and Equipment.2', 'Category 2: Labor and Equipment.3', 'Category 2: Labor and Equipment.4', 'Category 2: Labor and Equipment.5', 'Category 2: Labor and Equipment.6', 'Category 2: Labor and Equipment.7', 'Category 3: Financing', 'Category 3: Financing.1', 'Category 3: Financing.2', 'Category 3: Financing.3', 'Category 3: Financing.4', 'Category 4: Design and Documentation', 'Category 4: Design and Documentation.1', 'Category 4: Design and Documentation.2', 'Category 4: Design and Documentation.3', 'Category 4: Design and Documentation.4', 'Category 4: Design and Documentation.5', 'Category 5: Management and Organization', 'Category 5: Management and Organization .1', 'Category 5: Management and Organiza

In [52]:
print("numeric_columns:", numeric_columns)
print("Columns in df_encoded:", df_encoded.columns.tolist())

missing = [col for col in numeric_columns if col not in df_encoded.columns]
print("Missing numeric columns:", missing)


numeric_columns: []
Columns in df_encoded: ['Respondent Number', 'Category 1: Materials', 'Category 1: Materials.1', 'Category 1: Materials.2', 'Category 1: Materials.3', 'Category 1: Materials.4', 'Category 2: Labor and Equipment', 'Category 2: Labor and Equipment.1', 'Category 2: Labor and Equipment.2', 'Category 2: Labor and Equipment.3', 'Category 2: Labor and Equipment.4', 'Category 2: Labor and Equipment.5', 'Category 2: Labor and Equipment.6', 'Category 2: Labor and Equipment.7', 'Category 3: Financing', 'Category 3: Financing.1', 'Category 3: Financing.2', 'Category 3: Financing.3', 'Category 3: Financing.4', 'Category 4: Design and Documentation', 'Category 4: Design and Documentation.1', 'Category 4: Design and Documentation.2', 'Category 4: Design and Documentation.3', 'Category 4: Design and Documentation.4', 'Category 4: Design and Documentation.5', 'Category 5: Management and Organization', 'Category 5: Management and Organization .1', 'Category 5: Management and Organiza

In [53]:
df_encoded.select_dtypes(include=["int64", "float64"]).columns.tolist()


['Respondent Number']

In [54]:
['Planned_Duration', 'Actual_Duration', 'Budget']


['Planned_Duration', 'Actual_Duration', 'Budget']

In [55]:
numeric_columns = [
    'Planned_Duration',
    'Actual_Duration',
    'Budget'
]


In [56]:
numeric_columns = ["Respondent Number"]

df_scaled, scaler_metadata = scale_numerical(
    df_encoded,
    numeric_columns=numeric_columns
)

print("Numerical scaling completed.")
df_scaled.head()


Numerical scaling completed.


,Respondent Number,Category 1: Materials,Category 1: Materials.1,Category 1: Materials.2,Category 1: Materials.3,Category 1: Materials.4,Category 2: Labor and Equipment,Category 2: Labor and Equipment.1,Category 2: Labor and Equipment.2,Category 2: Labor and Equipment.3,...,Category 8: Scope of work.3,Category 9: External issues,Category 9: External issues .1,Category 9: External issues .2,Respondant Information,Respondant Information.1,Respondant Information.2,Respondant Information.3,Respondant Information.4,Respondant Information.5
1,-1.719981,1,1,1,1,1,1,1,1,1,...,1,2,2,3,1,2,2,2,3,5
2,-1.695756,2,5,3,2,3,2,2,2,3,...,3,1,1,4,1,3,2,3,3,3
3,-1.671530,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,2,1,2,4
4,-1.647305,2,2,3,3,2,2,3,4,4,...,3,4,3,3,1,3,1,1,3,4
5,-1.623080,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,3,1,2,4


In [57]:
[col for col in df_scaled.columns if "delay" in col.lower()]


[]

In [58]:
df_scaled.columns.tolist()


['Respondent Number',
 'Category 1: Materials',
 'Category 1: Materials.1',
 'Category 1: Materials.2',
 'Category 1: Materials.3',
 'Category 1: Materials.4',
 'Category 2: Labor and Equipment',
 'Category 2: Labor and Equipment.1',
 'Category 2: Labor and Equipment.2',
 'Category 2: Labor and Equipment.3',
 'Category 2: Labor and Equipment.4',
 'Category 2: Labor and Equipment.5',
 'Category 2: Labor and Equipment.6',
 'Category 2: Labor and Equipment.7',
 'Category 3: Financing',
 'Category 3: Financing.1',
 'Category 3: Financing.2',
 'Category 3: Financing.3',
 'Category 3: Financing.4',
 'Category 4: Design and Documentation',
 'Category 4: Design and Documentation.1',
 'Category 4: Design and Documentation.2',
 'Category 4: Design and Documentation.3',
 'Category 4: Design and Documentation.4',
 'Category 4: Design and Documentation.5',
 'Category 5: Management and Organization',
 'Category 5: Management and Organization .1',
 'Category 5: Management and Organization .2',
 'Cate

In [59]:
if target_column not in df_scaled.columns:
    candidates = [
        col for col in df_scaled.columns
        if any(key in col.lower() for key in ["delay", "risk", "overrun", "respondant information"])
    ]
    raise ValueError(
        f"Configured target_column '{target_column}' is not in df_scaled. "
        f"Possible candidates: {candidates}"
    )

X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
    df_scaled,
    target_column=target_column,
    test_size=test_size,
    val_size=val_size,
    random_state=random_state,
    stratify=use_stratify
)

print("Dataset split completed.")


Dataset split completed.


In [60]:
print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)


Training set: (114, 50) (114,)
Validation set: (3, 50) (3,)
Test set: (26, 50) (26,)


In [61]:
print("Preparation pipeline completed successfully.")


Preparation pipeline completed successfully.
